# Advanced Model Optimisation Exercises

In the following exercises, you will apply advanced modeling techniques to a patient dataset, allowing you to predict disease type, severity, and treatment outcomes.

## 📌 Table of Contents

1. [Data Preparation](#1)
2. [Cross Validation](#2)
3. [Hyperparameter Tuning](#3)
4. [Pruning](#4)
5. [Missing Data Handeling](#5)
6. [Feature Engineering](#6)
7. [Feature Selection](#7)
8. [Regularisation](#8)
9. [MLflow](#9)

**Importing packages**

In [ ]:
import os
import sys

sys.path.append(os.path.abspath(".."))
sys.path.append(os.path.abspath("../.."))

import numpy as np
import pandas as pd
from scipy.stats import randint, uniform
import matplotlib.pyplot as plt

import sklearn
from sklearn.model_selection import (
    KFold,
    StratifiedKFold,
    GridSearchCV,
    RandomizedSearchCV,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif, RFECV
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    r2_score,
    mean_absolute_error,
    mean_squared_error,
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

import xgboost
from xgboost import XGBClassifier

import optuna
from optuna.integration import OptunaSearchCV
from optuna.distributions import IntDistribution, FloatDistribution
from optuna.samplers import TPESampler, NSGAIISampler
from optuna.pruners import SuccessiveHalvingPruner

import mlflow
from badge_3203.mlflow import (
    start_mlflow,
    stop_mlflow,
    start_experiment,
    log_cross_validation_to_mlflow,
    log_search_to_mlflow,
)

from badge_3203.evaluate import evaluate_model


## 1. Data Preparation <a class="anchor" id="1"></a>

Note: you may simple run these cells and start at Chapter 2: Crossvalidation and start at exercise a). Do make sure to briefly inspect the data frame resulting from these steps so you know what you are working with. 

**Loading and Inspecting data**

In [ ]:
df = pd.read_csv("../Data/raw/disease.csv")

In [ ]:
df.shape

In [ ]:
df.head(10)

**Inspecting categorical variables**

In [ ]:
for col in df.select_dtypes(include=["object", "category"]).columns:
    print("\n" + "-"*40)
    print(f"Column: {col}")
    print(f"Number of unique values: {df[col].nunique(dropna=False)}")
    print("Value counts:")
    print(df[col].value_counts(dropna=False))

**Inspecting Numerical Variables**

In [ ]:
numeric_cols = df.select_dtypes(include="number").columns
numeric_cols = numeric_cols.drop("Patient_ID", errors="ignore")

df[numeric_cols].describe()


**Disentangeling Blood Pressure**

In [ ]:
df[["BP_systolic", "BP_diastolic"]] = (
    df["Blood_Pressure_mmHg"]
    .str.split("/", expand=True)
    .astype(float)
)


In [ ]:
df = df.drop(columns="Blood_Pressure_mmHg")

**Dummifying categorical variables**

In [ ]:
cat_cols = df.select_dtypes(include=["object", "category"]).columns

dummies = pd.get_dummies(
    df[cat_cols],
    drop_first=True
)

df = pd.concat([df, dummies], axis=1)

df = df.replace({True: 1, False: 0})

In [ ]:
df.head(50)

**Creating Binary Variable to serve as clinically meaningful targets**

In [ ]:
df["disease"] = (df["Diagnosis"] != "Healthy").astype(int)


In [ ]:
df["Hospitalisation"] = (df["Treatment_Plan"] == "Hospitalization and medication").astype(int)

In [ ]:
df["high_risk_case"] = (
    df["Severity"].isin(["Severe", "Moderate"]) &
    (df["Diagnosis"] == "Pneumonia")
).astype(int)

In [ ]:
df["high_risk_case"].value_counts()


**Cleaned and prepared Data**

In [ ]:
df.head(50)

**Defining Target and Predictors**

There are different things you can predict within this dataset that might be of clinical interest. Choose which of the following interests you the most 

- Predicting whether the patient is healthy or gets diagnosed with an illness
- Predicting whether the patient has an illness of severe severity
- Predicting whether the patient will need Hospitalisation

Specify your taregt and prediction variable accordingly

In [ ]:
# Define target
y = df["Severity_Severe"]

# Keep only numerical features
X = df.select_dtypes(include="number").copy()

# Remove target and other leakage columns (e.g. that carry direct info about your taregt that you wouldn't be able to access in a realistic model employment setting) and Patient ID 
leakage_cols = [
    col for col in X.columns
    if col.startswith("Diagnosis_")
    or col.startswith("Severity")
    or col.startswith("Treatment")
    or col.startswith("hospital")
]

# Drop target, ID, and leakage-related columns
X = X.drop(columns=["disease", "Patient_ID", "high_risk_case", "Hospitalisation"] + leakage_cols, errors="ignore")

# Print features used for training 
print("Remaining feature columns:")
print(X.columns)

# 2. Cross-validation <a class="anchor" id="2"></a>

# Exercise a) implementing simple cross-validation

For the first exercise, we will optimize the hyperparameters for our xgboost model. You can find more information about xgboost and its different parameters in the [documentation](https://xgboost.readthedocs.io/en/stable/parameter.html) and some more guidance on [tuning](https://xgboost.readthedocs.io/en/stable/tutorials/param_tuning.html). 

Decide on the hyperparameters you would like to try out. We would recommend keeping the search space smaller than 30 options to prevent long run times. 

In [ ]:
# Define XGboost model
xgb = 
# Specify hyperparameter grid

param_grid = {
 
}

# 5 fold cross validation
cv = 

grid = GridSearchCV(
   
)

# Run Gridsearch
grid.fit(X, y)

# Results

## Exercise b) Implementing nested cross-validation

In this exercise, we go one step further by selecting not only the best hyperparameters but also the best model. To do this, we use nested cross-validation, where the inner loop tunes hyperparameters and the outer loop provides an unbiased estimate of each model’s performance. This separation ensures an unbiased performance estimate and allows us to reliably compare different model types. For your comparison, focus on xgboost, decision tree and random forest. 

In [ ]:

# Define models and hyperparameter grids 

models_and_grids = {
    
}

# Nested CV setup:
# - Outer CV: unbiased evaluation
# - Inner CV: hyperparameter tuning for each model
outer_cv = 
inner_cv = 



# 3. Hyperparameter Tuning <a class="anchor" id="3"></a>

## Exercise c) Implementing Random Search for hyperparameter tuning

In the previous exercises we used grid search for hyperparameter tuning. However, we've already learned that a random search is often more efficient than a grid search, so let's implement it!


Unlike grid search, you'll define the parameter space using distributions to sample from. If you supply an array it will be sampled uniformly. You can also use other distributions, like loguniform, which is often used for parameters like learning_rate.
See [scipy distributions](https://docs.scipy.org/doc/scipy/reference/stats.html) for a full list of distributions you can use.

Define your search space using these distributions.

Unlike grid search, the parameter space for a random search does not define the exact number of trials. You need to decide how many iterations to run based on how much time you want to invest. Remember, each trial fits multiple models for cross-validation, so we recommend keeping the number of trials under 30 to keep the runtime quick enough.


In [ ]:
xgb = 

param_distributions = {
  
}

# 5-fold cross validation

cv = 


# Random search with CV

random_search = 


# Results 


## Exercise d) Different optimization techniques: Optuna - Bayesian
As you now have already seen, there are different optimization techniques with different structures. We'll now be using Optuna which implements some more complex hyperparameter search algorithms. You can find more information about the different algorithms in the documentation: [optuna algorithms](https://optuna.readthedocs.io/en/stable/tutorial/10_key_features/003_efficient_optimization_algorithms.html). The standard algorithm is a Tree-structured Parzen Estimator (TPE), which is a type of bayesian estimator.

Optuna implements a similar cross-validation hyperparameter optimization function as sklearn: [OptunaSearchCV](https://optuna.readthedocs.io/en/v2.0.0/reference/generated/optuna.integration.OptunaSearchCV.html). However, the parameter distributions are defined slightly differently, see [optuna distributions](https://optuna.readthedocs.io/en/stable/reference/distributions.html) for the available distributions.

In [ ]:
# Model 
xgb = 

# Optuna (Bayesian/TPE) search space 
param_distributions = {
   
}

# 5-fold CV 
cv = 

# Bayesian optimization via OptunaSearchCV (default sampler is TPE) 
optuna_search = 




## Exercise e) Different optimization techniques in Optuna: Genetic Optimization

As mentioned, Optuna also supports other optimization algorithms, including a genetic algorithm. You can implement this by specifying a sampler in the optimization process

You can add any specifications for the genetic sampler here: [NSGAIISampler](https://optuna.readthedocs.io/en/stable/reference/samplers/generated/optuna.samplers.NSGAIISampler.html). 

In [ ]:
# Model
xgb = 

# Optuna search space
param_distributions = {
   
}

# 5-fold CV
cv = 

# Genetic sampler (NSGA-II)
sampler = 


# OptunaSearchCV using the genetic sampler
optuna_search = 

# Run the hyperparameter optimization
optuna_search.fit(X, y)

# 4. Pruning <a class="anchor" id="4"></a>

## Exercise f) Pruning the optimization for efficiency 
Now it’s time to speed up our hyperparameter optimization by pruning away unpromising trials early, so the strongest configurations get more compute.
Implement Optuna with pruning using the SuccessiveHalvingPruner. Take not of how many trials were pruned versus completed. W

In [ ]:
# Define the XGBoost model
xgb = 

# Define the hyperparameter search space
param_distributions = {
    
}

# Define the Optuna sampler and successive halving pruner
sampler = 
pruner =


# Set up OptunaSearchCV with pruning enabled
optuna_search = 
)

# Run the hyperparameter optimization
optuna_search.fit(X, y)


# 5. Missing Data Handeling <a class="anchor" id="5"></a>

In most cases, the data is not as complete as the one we have been wroking with in these exercises. Many times, there will be missing data. In this following exercise we will therefore work with the same dataset, however, now containing missing values in multiple columns. 

In [ ]:
df_nan = pd.read_csv("../data/raw/disease_nan.csv")

In [ ]:
# Missing values per column
missing_summary = pd.DataFrame({
    "missing_count": df_nan.isna().sum(),
    "missing_percent": df_nan.isna().mean() * 100
}).sort_values("missing_count", ascending=False)

missing_summary


## Exercise g) identify the missingness mechanism 

Use the tools discussed in the presentation to figure out what the underlying mechnisms of the missing data may be

**Analysis of Oxigen Saturation Missingness**

In [ ]:
# Put your code here...

**Analysis of Heart rate missingness**

In [ ]:
# Put your code here...

**Analysis of Age Missingness**

In [ ]:
# Put your code here...

In [ ]:
# HELPER FUNCTION TO PERFORM LITTLE'S MCAR TEST
# Source - https://stackoverflow.com/a
# Posted by Sadegh
# Retrieved 2026-01-09, License - CC BY-SA 4.0

from scipy.stats import chi2

def little_mcar_test(data, alpha=0.05):
    """
    Performs Little's MCAR (Missing Completely At Random) test on a dataset with missing values.
    
    Parameters:
    data (DataFrame): A pandas DataFrame with n observations and p variables, where some values are missing.
    alpha (float): The significance level for the hypothesis test (default is 0.05).
    
    Returns:
    A tuple containing:
    - A matrix of missing values that represents the pattern of missingness in the dataset.
    - A p-value representing the significance of the MCAR test.
    """
    
    # Calculate the proportion of missing values in each variable
    p_m = data.isnull().mean()
    
    # Calculate the proportion of complete cases for each variable
    p_c = data.dropna().shape[0] / data.shape[0]
    
    # Calculate the correlation matrix for all pairs of variables that have complete cases
    R_c = data.dropna().corr()
    
    # Calculate the correlation matrix for all pairs of variables using all observations
    R_all = data.corr()
    
    # Calculate the difference between the two correlation matrices
    R_diff = R_all - R_c
    
    # Calculate the variance of the R_diff matrix
    V_Rdiff = np.var(R_diff, ddof=1)
    
    # Calculate the expected value of V_Rdiff under the null hypothesis that the missing data is MCAR
    E_Rdiff = (1 - p_c) / (1 - p_m).sum()
    
    # Calculate the test statistic
    T = np.trace(R_diff) / np.sqrt(V_Rdiff * E_Rdiff)
    
    # Calculate the degrees of freedom
    df = data.shape[1] * (data.shape[1] - 1) / 2
    
    # Calculate the p-value using a chi-squared distribution with df degrees of freedom and the test statistic T
    p_value = 1 - chi2.cdf(T ** 2, df)
    
    # Create a matrix of missing values that represents the pattern of missingness in the dataset
    missingness_matrix = data.isnull().astype(int)
    
    # Return the missingness matrix and the p-value
    return missingness_matrix, p_value



## Exercise h) decide how to handle the missing data

Based on your hypotheses of exercise g) select an appropriate method for handeling the missing data. 

# 6. Feature engineering <a class="anchor" id="6"></a>

## Exercise i) Feature Engineering

We've discussed the importance of feature engineering. Now, think about any new features you'd like to introduce to enhance your model. Consider transformations, interactions between existing features, or new derived variables that could improve the model's predictive power.

Be creative! 

After you finished your feature engineering save the dataframe as a csv to Data/processed with an informative name! This way you can retrieve your augmented dataset easily for future experimentation. 

In [ ]:
# Put your code here...

**Saving Dataframe to processed folder**

In [ ]:
df.to_csv("../Data/processed/disease_processed.csv", index=False)


# 7. Feature Selection <a class="anchor" id="7"></a>

For the following two exercises, we'll only focus on feature selection. Use the best hyperparameters you found in the previous exercises with your xgboost model. We start applying the methods using a classic train-test split. 

**Train-Test Split**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
)

**Defining our model**

In [ ]:
# XGBoost model with your best hyperparameters ADAPT HYPERPARAMS TO YOUR BEST
model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
    colsample_bytree=1.0,
    learning_rate=0.05,
    max_depth=5,
    n_estimators=200,
    subsample=1.0,
)

## Exercise j) Filter methods 

There are many possible [Sklearn filter methods](https://scikit-learn.org/stable/modules/feature_selection.html#univariate-feature-selection) to select features. For this exercise, let's use the [f_classif](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.f_classif.html) function to score features based on their correlation with the target variable. We'll then select the top 6 features with the highest scores to include in the model, ensuring that we focus on the most relevant predictors

In [ ]:
# Feature selection method, use a Filter method.
feature_selection_filter = ...

We can assess how stable our feature selection is using cross-validation. This will help us study how well the selected features perform across different subsets of the data.

In [ ]:
outer_cv = ...


In [ ]:


# Count how often each feature is selected
print("How many times is each feature selected")
....

**Questions**
1. What are the main weaknesses of a filter method?
2. Do you notice anything about the selected features? (what could be going wrong?)
3. How many features would you like to select?

**Bonus**
1. Can you solve the problem you found in question 2?
2. How would you add hyperparameter optimization to this pipeline?

## Exercise k) Wrapper methods 

Now, let's implement Recursive Feature Elimination (RFE) with cross-validation as our wrapper method. This method recursively removes the least important features and evaluates model performance using cross-validation at each step. It allows us to identify the optimal subset of features that maximizes model performance.

We can use [RFECV()](https://scikit-learn.org/1.5/modules/generated/sklearn.feature_selection.RFECV.html) to implement this. Like SelectKBest, RFECV returns a reduced feature set, which is passed to the next step in the pipeline. Therefore, you need to place the model behind the RFECV in the pipeline, but also pass the model as a parameter to RFECV for it to use in the feature selection process.

In [ ]:
# Wrapper method using RFECV
feature_selection_wrapper = ...


We can assess how stable our feature selection is using cross-validation. This will help us study how well the selected features perform across different subsets of the data.

In [ ]:
# It might take a little bit to run these due to the large number of features
# You can speed up the process by only using a subset of the features when loading the data at the top of this notebook
outer_cv = ...


In [ ]:

# Count how often each feature is selected
print("How many times is each feature selected")

...

**Questions**
1. Why would we prefer this method over sequential feature selection

# 8. Regularisation <a class="anchor" id="8"></a>

## Exercise l) Regularisation

To make a comparison, implement one xgboost model without regularisation, one with L1 regularisaton, one with L2 regularisation and one with early stopping. 

What do you obsere in terms of final performance, train-validation difference? 
Also, plot feature importance for the different models and report what you observe there. Are the results as expected? 

In [ ]:
# Models ADAPT TO YOUR HYPERPARAMS
base_params = dict(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
    learning_rate=0.05,
    max_depth=5,
    subsample=1.0,
    colsample_bytree=1.0,
    n_estimators=1000
)

model_none = ...
model_l1 = ...
model_l2 = ...
model_es = ...




# 9. ML Flow <a class="anchor" id="9"></a>

## Exercise m) Structuring experiments using ML flow 

In this exercise, you will use MLflow to log the results of a simple 5-fold cross-validation experiment.
Define your target and predictor variables and set up a model using the hyperparameters you previously identified. Evaluate the model’s performance using 5-fold cross-validation, and log fold-level performance as well as the mean and standard deviation of the evaluation metric using MLflow.

In [ ]:
START_SERVER = True

process = None
if START_SERVER:
    process = start_mlflow()

# Load Processed Data
df = pd.read_csv(...)


# Define target and features

y = 

X = 

# drop leakage columns/prefixes + target itself
drop_prefixes = ("Diagnosis", "Treatment", "Severity")
cols_to_drop = [c for c in X.columns if c.startswith(drop_prefixes)]
cols_to_drop += ["high_risk_case"]

X = X.drop(columns=cols_to_drop, errors="ignore")


# Set up MLflow experiment 
start_experiment("Exercise_CV_Basics")


# Cross validation + logging
cv = ...

with mlflow.start_run(run_name="xgb_cv_baseline"):
...

# stop server
if process is not None:
    stop_mlflow(process)
